# 📊 Experiment1: Comprehensive Benchmarking Analysis

**Objective**: Systematic evaluation and comparison of machine learning methods for credit risk prediction across multiple datasets with and without hyperparameter optimization.

---

## 📋 Overview

This notebook presents a comprehensive analysis of **Experiment1**, which evaluates selected methods from Experiment0 across two critical credit risk tasks:

### Tasks
1. **PD (Probability of Default)**: Binary classification predicting borrower default
2. **LGD (Loss Given Default)**: Regression estimating loss severity upon default

### Experimental Design
Each method is evaluated in two configurations:
- **NO_HPO**: Default hyperparameters (baseline)
- **HPO**: Hyperparameters optimized via Optuna (tuned)

Multiple datasets are used for each task with 5-fold cross-validation, ensuring robust performance estimates.

---

## 🔬 Analysis Components

This notebook provides four main analysis perspectives:

### Part A: PD Task Analysis (Classification)
- **A1. Performance Heatmaps**: Dataset × Method matrices showing AUC scores
- **A2. Performance Distributions**: Bar charts and boxplots across folds
- **A3. Rank Analysis**: Method rankings with heatmaps and average ranks
- **A4. PAMA Analysis**: Probability of Achieving Maximal Accuracy (fold-level)

### Part B: LGD Task Analysis (Regression)
- **B1. Performance Heatmaps**: Dataset × Method matrices showing R² scores
- **B2. Performance Distributions**: Bar charts and boxplots across folds
- **B3. Rank Analysis**: Method rankings with heatmaps and average ranks
- **B4. PAMA Analysis**: Probability of Achieving Maximal Accuracy (fold-level)

### Part C: Dataset Characteristics Analysis
- **C1. Load Dataset Characteristics**: Extract size, features, dimensionality
- **C2. PD Rank Correlations**: Spearman correlations with dataset properties
- **C3. LGD Rank Correlations**: Identify which methods scale better

### Part D: Training Time Analysis
- **D1. PD Training Times**: Heatmaps and average times per method
- **D2. LGD Training Times**: Computational efficiency comparison
- **D3. HPO Impact on Time**: Additional computational cost of tuning

---

## 📊 Key Visualizations

For each task and analysis type, the notebook generates:
- **Heatmaps**: Color-coded matrices for easy pattern recognition
- **Bar Charts**: Average performance with error bars
- **Boxplots**: Distribution analysis across folds
- **Correlation Plots**: Method behavior vs dataset characteristics
- **PAMA Charts**: Win rate visualization

All visualizations are saved at 300 DPI in the `figures/` directory.

---

## 🎯 Research Questions Addressed

1. **Performance**: Which methods achieve the best predictive accuracy?
2. **HPO Impact**: How much does hyperparameter tuning improve results?
3. **Consistency**: Which methods perform reliably across datasets?
4. **Scalability**: How do methods perform on larger/more complex datasets?
5. **Efficiency**: What is the computational cost of each method?
6. **Specialization**: Do certain methods excel on specific dataset characteristics?

---

## 1. Setup & Configuration

In [ ]:
# Standard imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
import math

# =============================================================================
# SETUP PATHS (Notebook is in notebooks/ folder)
# =============================================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT_NAME = "experiment1"
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME
SUMMARY_DIR = RESULTS_DIR / "summary"
FIGURES_DIR = RESULTS_DIR / "figures"

print("=" * 80)
print("  EXPERIMENT1 COMPREHENSIVE ANALYSIS")
print("=" * 80)
print(f"📂 Project root:       {PROJECT_ROOT}")
print(f"📂 Results directory:  {RESULTS_DIR}")
print(f"📂 Summary directory:  {SUMMARY_DIR}")
print(f"📂 Figures directory:  {FIGURES_DIR}")

# =============================================================================
# GENERATE SUMMARIES IF NOT ALREADY PRESENT
# =============================================================================

if SUMMARY_DIR.exists() and any(SUMMARY_DIR.glob("*.csv")):
    print(f"\n✓ Summary files already exist. Skipping generation.")
    print(f"   (Delete {SUMMARY_DIR} manually to regenerate)")
else:
    print(f"\n📊 Summary files not found. Generating summaries...")
    from src.utils.summarize_results import summarize_results
    print("\n" + "=" * 80)
    summarize_results(experiment=EXPERIMENT_NAME)
    print("=" * 80)

# Ensure figures directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ Setup complete! Ready for analysis.")
print("=" * 80)

In [ ]:
# Plotting configuration
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Color schemes
CMAP_PERFORMANCE = 'RdYlGn'  # Red (bad) to Green (good)
CMAP_TIME = 'YlOrRd'  # Yellow (fast) to Red (slow)

## 2. Data Loading

In [ ]:
# Load raw and aggregated results
pd_raw = pd.read_csv(SUMMARY_DIR / "summary_pd_raw.csv")
pd_agg = pd.read_csv(SUMMARY_DIR / "summary_pd_aggregated.csv")
lgd_raw = pd.read_csv(SUMMARY_DIR / "summary_lgd_raw.csv")
lgd_agg = pd.read_csv(SUMMARY_DIR / "summary_lgd_aggregated.csv")

print("\n📊 Data Overview:")
print(f"\nPD (Probability of Default):")
print(f"  - Raw results: {len(pd_raw)} fold results")
print(f"  - Methods: {pd_raw['method'].nunique()}")
print(f"  - Datasets: {pd_raw['dataset'].nunique()}")
print(f"  - HPO modes: {sorted(pd_raw['hpo_mode'].unique())}")
print(f"  - Methods: {sorted(pd_raw['method'].unique())}")

print(f"\nLGD (Loss Given Default):")
print(f"  - Raw results: {len(lgd_raw)} fold results")
print(f"  - Methods: {lgd_raw['method'].nunique()}")
print(f"  - Datasets: {lgd_raw['dataset'].nunique()}")
print(f"  - HPO modes: {sorted(lgd_raw['hpo_mode'].unique())}")
print(f"  - Methods: {sorted(lgd_raw['method'].unique())}")

## 3. Helper Functions

In [ ]:
def create_performance_heatmap(df, metric, hpo_mode, task_name, cmap='RdYlGn', figsize=(24, 12)):
    """
    Create a heatmap showing method performance across datasets.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"⚠️  Metric '{metric}' not found in data")
        return None, None
    
    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()
    
    if df_filtered.empty:
        print(f"⚠️  No data for {hpo_mode}")
        return None, None
    
    pivot = df_filtered.pivot(index='dataset', columns='method', values=mean_col)
    pivot = pivot.sort_index()
    method_means = pivot.mean(axis=0).sort_values(ascending=False)
    pivot = pivot[method_means.index]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    if metric == 'R2':
        vmin = pivot.min().min()
        vmax = pivot.max().max()
        abs_max = max(abs(vmin), abs(vmax))
        
        sns.heatmap(
            pivot, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, vmin=-abs_max, vmax=abs_max,
            cbar_kws={'label': metric}, linewidths=0.5, ax=ax
        )
    else:
        vmin = pivot.min().min()
        vmax = pivot.max().max()
        
        sns.heatmap(
            pivot, annot=True, fmt='.3f', cmap=cmap,
            vmin=vmin, vmax=vmax,
            cbar_kws={'label': metric}, linewidths=0.5, ax=ax
        )
    
    ax.set_title(f'{task_name} Performance: {metric} ({hpo_mode})\nDatasets × Methods', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_heatmap_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    return fig, pivot


def create_rank_heatmap(df, metric, hpo_mode, task_name, figsize=(24, 12)):
    """
    Create a heatmap showing method ranks across datasets (1=best, n=worst).
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"⚠️  Metric '{metric}' not found in data")
        return None, None
    
    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()
    
    if df_filtered.empty:
        print(f"⚠️  No data for {hpo_mode}")
        return None, None
    
    pivot = df_filtered.pivot(index='dataset', columns='method', values=mean_col)
    rank_pivot = pivot.rank(axis=1, ascending=False, method='average')
    rank_pivot = rank_pivot.sort_index()
    method_avg_ranks = rank_pivot.mean(axis=0).sort_values()
    median_ranks = rank_pivot.median(axis=0).sort_values()
    rank_pivot = rank_pivot[method_avg_ranks.index]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        rank_pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
        cbar_kws={'label': 'Rank (1=best)'}, linewidths=0.5, ax=ax
    )
    
    ax.set_title(f'{task_name} Ranks: {metric} ({hpo_mode})\nDatasets × Methods (1=best)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_ranks_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    return fig, rank_pivot, method_avg_ranks, median_ranks


def create_hpo_improvement_heatmap(df, metric, task_name, figsize=(24, 12)):
    """
    Create a heatmap showing improvement from NO_HPO to HPO.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"⚠️  Metric '{metric}' not found in data")
        return None, None
    
    no_hpo = df[df['hpo_mode'] == 'NO_HPO'].pivot(index='dataset', columns='method', values=mean_col)
    hpo = df[df['hpo_mode'] == 'HPO'].pivot(index='dataset', columns='method', values=mean_col)
    
    diff = hpo - no_hpo
    diff = diff.sort_index()
    method_avg_improvement = diff.mean(axis=0).sort_values(ascending=False)
    diff = diff[method_avg_improvement.index]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    vmax = max(abs(diff.min().min()), abs(diff.max().max()))
    
    sns.heatmap(
        diff, annot=True, fmt='+.3f', cmap='RdYlGn',
        center=0, vmin=-vmax, vmax=vmax,
        cbar_kws={'label': f'{metric} Improvement (HPO - NO_HPO)'},
        linewidths=0.5, ax=ax
    )
    
    ax.set_title(f'{task_name}: HPO Impact on {metric}\nDatasets × Methods (green=improvement, red=degradation)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    filename = FIGURES_DIR / f"{task_name.lower()}_hpo_improvement_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    n_improvements = (diff > 0).sum().sum()
    n_degradations = (diff < 0).sum().sum()
    n_total = diff.notna().sum().sum()
    
    print(f"\n📊 HPO Impact Statistics:")
    print(f"   Improvements: {n_improvements}/{n_total} ({100*n_improvements/n_total:.1f}%)")
    print(f"   Degradations: {n_degradations}/{n_total} ({100*n_degradations/n_total:.1f}%)")
    print(f"   Avg improvement: {diff.mean().mean():+.4f}")
    print(f"   Max improvement: {diff.max().max():+.4f}")
    print(f"   Max degradation: {diff.min().min():+.4f}")
    
    return fig, diff


def plot_average_performance_bar(df, metric, hpo_mode, task_name, figsize=(20, 8)):
    """
    Bar chart showing average performance across datasets with error bars.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"⚠️  Metric '{metric}' not found in data")
        return None
    
    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()
    
    if df_filtered.empty:
        print(f"⚠️  No data for {hpo_mode}")
        return None
    
    method_avg = df_filtered.groupby('method')[mean_col].mean().sort_values(ascending=False)
    method_std = df_filtered.groupby('method')[mean_col].std()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    x = np.arange(len(method_avg))
    bars = ax.bar(x, method_avg.values, yerr=method_std.loc[method_avg.index].values,
                   capsize=5, alpha=0.8, edgecolor='black', linewidth=1.2)
    
    norm = plt.Normalize(vmin=method_avg.min(), vmax=method_avg.max())
    colors = plt.cm.RdYlGn(norm(method_avg.values))
    for bar, color in zip(bars, colors):
        bar.set_facecolor(color)
    
    ax.set_xticks(x)
    ax.set_xticklabels(method_avg.index, rotation=45, ha='right')
    ax.set_ylabel(f'{metric} (Mean ± Std)', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name}: Average {metric} Across Datasets ({hpo_mode})',
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    
    for i, (val, std) in enumerate(zip(method_avg.values, method_std.loc[method_avg.index].values)):
        ax.text(i, val + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_bar_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    return fig


def plot_performance_distribution(df, metric, hpo_mode, task_name, figsize=(20, 8)):
    """
    Boxplot showing performance distribution across all folds.
    """
    if metric not in df.columns:
        print(f"⚠️  Metric '{metric}' not found in raw data")
        return None
    
    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()
    
    if df_filtered.empty:
        print(f"⚠️  No data for {hpo_mode}")
        return None
    
    method_order = df_filtered.groupby('method')[metric].median().sort_values(ascending=False).index
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.boxplot(
        data=df_filtered, x='method', y=metric, order=method_order,
        palette='Set2', ax=ax
    )
    
    sns.stripplot(
        data=df_filtered, x='method', y=metric, order=method_order,
        color='black', alpha=0.3, size=4, ax=ax
    )
    
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name}: {metric} Distribution Across All Folds ({hpo_mode})',
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_boxplot_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    return fig

def plot_rank_bar(avg_ranks, median_ranks, task_name, metric, hpo_mode, figsize=(20, 8)):
    """
    Bar chart showing average ranks.
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    x = np.arange(len(avg_ranks))
    bars = ax.bar(x, avg_ranks.values, alpha=0.8, edgecolor='black', linewidth=1.2)
    
    # Reverse colormap: low rank (good) = green, high rank (bad) = red
    norm = plt.Normalize(vmin=avg_ranks.min(), vmax=avg_ranks.max())
    colors = plt.cm.RdYlGn_r(norm(avg_ranks.values))
    for bar, color in zip(bars, colors):
        bar.set_facecolor(color)
    
    ax.set_xticks(x)
    ax.set_xticklabels(avg_ranks.index, rotation=45, ha='right')
    ax.set_ylabel('Average Rank (lower is better)', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name}: Average Ranks for {metric} ({hpo_mode})',
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    
    for i, val in enumerate(avg_ranks.values):
        # Average printed at +0.2
        ax.text(i, val + 0.2, f'Avg: {val:.2f}', ha='center', va='bottom', fontsize=9)
        
        # Median printed higher at +1.0 (was +0.5) to avoid overlap
        med_val = median_ranks[avg_ranks.index[i]]
        ax.text(i, val + 1.0, f'Med: {med_val:.2f}', ha='center', va='bottom', fontsize=8, style='italic', color='darkblue')
    
    plt.tight_layout()
    
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_bar_rank_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename.name}")
    
    return fig

---
# 📊 Part A: PD Analysis (Classification)

**Primary Metric**: AUC (Area Under ROC Curve)  
Range: 0.5-1.0, where 0.5 = random, 1.0 = perfect

## A1. Performance Heatmaps

### A1.1 NO_HPO Performance Matrix

In [ ]:
print("Creating PD AUC heatmap (NO_HPO)...")
fig_pd_no_hpo, pivot_pd_no_hpo = create_performance_heatmap(
    pd_agg, 'AUC', 'NO_HPO', 'PD', cmap='RdYlGn'
)
plt.show()

### A1.2 HPO Performance Matrix

In [ ]:
print("Creating PD AUC heatmap (HPO)...")
fig_pd_hpo, pivot_pd_hpo = create_performance_heatmap(
    pd_agg, 'AUC', 'HPO', 'PD', cmap='RdYlGn'
)
plt.show()

### A1.3 HPO Improvement Matrix

In [ ]:
print("Creating PD HPO improvement heatmap...")
fig_pd_improvement, diff_pd = create_hpo_improvement_heatmap(
    pd_agg, 'AUC', 'PD'
)
plt.show()

## A2. Performance Distributions

### A2.1 Average Performance Bar Charts

In [ ]:
print("Creating PD average performance bar chart (NO_HPO)...")
plot_average_performance_bar(pd_agg, 'AUC', 'NO_HPO', 'PD')
plt.show()

print("\nCreating PD average performance bar chart (HPO)...")
plot_average_performance_bar(pd_agg, 'AUC', 'HPO', 'PD')
plt.show()

### A2.2 Performance Distribution Boxplots

In [ ]:
print("Creating PD AUC distribution boxplot (NO_HPO)...")
plot_performance_distribution(pd_raw, 'AUC', 'NO_HPO', 'PD')
plt.show()

print("\nCreating PD AUC distribution boxplot (HPO)...")
plot_performance_distribution(pd_raw, 'AUC', 'HPO', 'PD')
plt.show()

## A3. Rank Analysis

### A3.1 Rank Heatmaps

In [ ]:
print("Creating PD rank heatmap (NO_HPO)...")
fig_pd_ranks_no_hpo, ranks_pd_no_hpo, avg_ranks_pd_no_hpo, median_ranks_pd_no_hpo = create_rank_heatmap(
    pd_agg, 'AUC', 'NO_HPO', 'PD'
)
plt.show()

print("\nMethod Rankings (NO_HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_pd_no_hpo.items(), 1):
    med_rank = median_ranks_pd_no_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

In [ ]:
print("Creating PD rank heatmap (HPO)...")
fig_pd_ranks_hpo, ranks_pd_hpo, avg_ranks_pd_hpo, median_ranks_pd_hpo = create_rank_heatmap(
    pd_agg, 'AUC', 'HPO', 'PD'
)
plt.show()

print("\nMethod Rankings (HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_pd_hpo.items(), 1):
    med_rank = median_ranks_pd_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

### A3.2 Average Rank Bar Charts

In [ ]:
if avg_ranks_pd_no_hpo is not None:
    print("Creating PD average rank bar chart (NO_HPO)...")
    plot_rank_bar(avg_ranks_pd_no_hpo, median_ranks_pd_no_hpo, 'PD', 'AUC', 'NO_HPO')
    plt.show()

if avg_ranks_pd_hpo is not None:
    print("\nCreating PD average rank bar chart (HPO)...")
    plot_rank_bar(avg_ranks_pd_hpo, median_ranks_pd_hpo, 'PD', 'AUC', 'HPO')
    plt.show()

## A4. Statistical Testing

### A4.1 PAMA Analysis

In [ ]:
print("\n" + "=" * 80)
print("  PD - PAMA ANALYSIS (HPO) - Fold-Level")
print("=" * 80)

# Use raw data to calculate PAMA at fold level
pd_hpo_raw = pd_raw[pd_raw['hpo_mode'] == 'HPO'].copy()

if not pd_hpo_raw.empty and 'AUC' in pd_hpo_raw.columns:
    pama_scores = {}
    
    # Group by dataset and fold to find winner for each fold
    for (dataset, fold), group in pd_hpo_raw.groupby(['dataset', 'fold_id']):
        if len(group) > 0 and 'AUC' in group.columns:
            max_score = group['AUC'].max()
            best_methods = group[group['AUC'] >= max_score - 0.0001]['method'].tolist()
            
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Total number of folds
    n_folds = len(pd_hpo_raw.groupby(['dataset', 'fold_id']))
    
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama_pct': 100 * score / n_folds}
        for method, score in pama_scores.items()
    ]).sort_values('pama_pct', ascending=False)
    
    print(f"\nPAMA Scores (across {n_folds} folds):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama_pct'] / 2)
        print(f"  {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama_pct']:5.1f}%  {bar}")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.RdYlGn(pama_df['pama_pct'] / pama_df['pama_pct'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=colors, edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'PD: Probability of Achieving Maximal Accuracy (HPO)\nFold-Level Analysis (n={n_folds} folds)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama_pct'] + 1, i, f"{row['pama_pct']:.1f}%", va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = FIGURES_DIR / "pd_pama_analysis_hpo.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {filename.name}")
    plt.show()
else:
    print("⚠️  No HPO data available for PAMA analysis")

---
# 📈 Part B: LGD Analysis (Regression)

**Primary Metric**: R² (Coefficient of Determination)  
Range: -∞ to 1.0, where 0 = baseline, 1.0 = perfect, <0 = worse than baseline

## B1. Performance Heatmaps

### B1.1 NO_HPO Performance Matrix

In [ ]:
print("Creating LGD R² heatmap (NO_HPO)...")
fig_lgd_no_hpo, pivot_lgd_no_hpo = create_performance_heatmap(
    lgd_agg, 'R2', 'NO_HPO', 'LGD', cmap='RdYlGn'
)
plt.show()

### B1.2 HPO Performance Matrix

In [ ]:
print("Creating LGD R² heatmap (HPO)...")
fig_lgd_hpo, pivot_lgd_hpo = create_performance_heatmap(
    lgd_agg, 'R2', 'HPO', 'LGD', cmap='RdYlGn'
)
plt.show()

### B1.3 HPO Improvement Matrix

In [ ]:
print("Creating LGD HPO improvement heatmap...")
fig_lgd_improvement, diff_lgd = create_hpo_improvement_heatmap(
    lgd_agg, 'R2', 'LGD'
)
plt.show()

## B2. Performance Distributions

### B2.1 Average Performance Bar Charts

In [ ]:
print("Creating LGD average performance bar chart (NO_HPO)...")
plot_average_performance_bar(lgd_agg, 'R2', 'NO_HPO', 'LGD')
plt.show()

print("\nCreating LGD average performance bar chart (HPO)...")
plot_average_performance_bar(lgd_agg, 'R2', 'HPO', 'LGD')
plt.show()

### B2.2 Performance Distribution Boxplots

In [ ]:
print("Creating LGD R² distribution boxplot (NO_HPO)...")
plot_performance_distribution(lgd_raw, 'R2', 'NO_HPO', 'LGD')
plt.show()

print("\nCreating LGD R² distribution boxplot (HPO)...")
plot_performance_distribution(lgd_raw, 'R2', 'HPO', 'LGD')
plt.show()

## B3. Rank Analysis

### B3.1 Rank Heatmaps

In [ ]:
print("Creating LGD rank heatmap (NO_HPO)...")
fig_lgd_ranks_no_hpo, ranks_lgd_no_hpo, avg_ranks_lgd_no_hpo, median_ranks_lgd_no_hpo = create_rank_heatmap(
    lgd_agg, 'R2', 'NO_HPO', 'LGD'
)
plt.show()

print("\nMethod Rankings (NO_HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_lgd_no_hpo.items(), 1):
    med_rank = median_ranks_lgd_no_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

In [ ]:
print("Creating LGD rank heatmap (HPO)...")
fig_lgd_ranks_hpo, ranks_lgd_hpo, avg_ranks_lgd_hpo, median_ranks_lgd_hpo = create_rank_heatmap(
    lgd_agg, 'R2', 'HPO', 'LGD'
)
plt.show()

print("\nMethod Rankings (HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_lgd_hpo.items(), 1):
    med_rank = median_ranks_lgd_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

### B3.2 Average Rank Bar Charts

In [ ]:
if avg_ranks_lgd_no_hpo is not None:
    print("Creating LGD average rank bar chart (NO_HPO)...")
    plot_rank_bar(avg_ranks_lgd_no_hpo, median_ranks_lgd_no_hpo, 'LGD', 'R2', 'NO_HPO')
    plt.show()

if avg_ranks_lgd_hpo is not None:
    print("\nCreating LGD average rank bar chart (HPO)...")
    plot_rank_bar(avg_ranks_lgd_hpo, median_ranks_lgd_hpo, 'LGD', 'R2', 'HPO')
    plt.show()

## B4. Statistical Testing

### B4.1. PAMA Analysis

In [ ]:
print("\n" + "=" * 80)
print("  LGD - PAMA ANALYSIS (HPO) - Fold-Level")
print("=" * 80)

# Use raw data to calculate PAMA at fold level
lgd_hpo_raw = lgd_raw[lgd_raw['hpo_mode'] == 'HPO'].copy()

if not lgd_hpo_raw.empty and 'R2' in lgd_hpo_raw.columns:
    pama_scores = {}
    
    # Group by dataset and fold to find winner for each fold
    for (dataset, fold), group in lgd_hpo_raw.groupby(['dataset', 'fold_id']):
        if len(group) > 0 and 'R2' in group.columns:
            max_score = group['R2'].max()
            best_methods = group[group['R2'] >= max_score - 0.0001]['method'].tolist()
            
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Total number of folds
    n_folds = len(lgd_hpo_raw.groupby(['dataset', 'fold_id']))
    
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama_pct': 100 * score / n_folds}
        for method, score in pama_scores.items()
    ]).sort_values('pama_pct', ascending=False)
    
    print(f"\nPAMA Scores (across {n_folds} folds):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama_pct'] / 2)
        print(f"  {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama_pct']:5.1f}%  {bar}")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.RdYlGn(pama_df['pama_pct'] / pama_df['pama_pct'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=colors, edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'LGD: Probability of Achieving Maximal Accuracy (HPO)\nFold-Level Analysis (n={n_folds} folds)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama_pct'] + 1, i, f"{row['pama_pct']:.1f}%", va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = FIGURES_DIR / "lgd_pama_analysis_hpo.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {filename.name}")
    plt.show()
else:
    print("⚠️  No HPO data available for PAMA analysis")

---
# 📐 Part C: Dataset Characteristics Analysis

Analyze the relationship between method rankings and dataset characteristics:
- **Dataset size** (number of rows)
- **Number of features** (columns)
- **Dimensionality** (rows × columns)

## C1. Load Dataset Characteristics

In [ ]:
print("\n" + "=" * 80)
print("  LOADING DATASET CHARACTERISTICS")
print("=" * 80)

# Find processed data directory
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# DataFeeder PCA settings (from data_feeder.py)
MAX_FEATURES_THRESHOLD = 100  # PCA triggered if features exceed this
PCA_TARGET_FEATURES = 99       # Target number of PCA components

print(f"\n📂 Looking for processed data in: {PROCESSED_DATA_DIR}")

if not PROCESSED_DATA_DIR.exists():
    print(f"⚠️  Processed data directory not found: {PROCESSED_DATA_DIR}")
    print("   Run preprocessing first to generate processed datasets!")
else:
    print(f"✓ Found processed data directory")

# Collect dataset characteristics
dataset_info = []

# Get unique datasets from results
all_datasets_pd = pd_raw['dataset'].unique()
all_datasets_lgd = lgd_raw['dataset'].unique()

print(f"\n📊 Extracting characteristics for {len(all_datasets_pd)} PD datasets and {len(all_datasets_lgd)} LGD datasets...")

# Function to get dataset characteristics
def get_dataset_characteristics(dataset_name, task):
    """
    Extract dataset characteristics from processed numpy arrays.
    
    Reflects the ACTUAL features used during training:
    - If total features > 100, PCA reduces to 99 components
    - Otherwise, uses original feature count
    
    The preprocessing pipeline stores datasets as:
    data/processed/{task}/{dataset}/
        ├── N.npy (numerical features) - shape: (n_samples, n_num_features)
        ├── C.npy (categorical features) - shape: (n_samples, n_cat_features)  
        ├── y.npy (target) - shape: (n_samples,)
        └── info.json (metadata)
    """
    if not PROCESSED_DATA_DIR.exists():
        return None
    
    # Build path: data/processed/{task}/{dataset}/
    task_dir = PROCESSED_DATA_DIR / task.lower() / dataset_name
    
    # Check if directory exists
    if not task_dir.exists():
        return None
    
    # Check if required files exist
    y_path = task_dir / "y.npy"
    if not y_path.exists():
        return None
    
    try:
        # Load target array to get number of rows and statistics
        y = np.load(y_path, allow_pickle=False)
        n_rows = len(y)
        
        # --- NEW: Calculate Class Imbalance for PD ---
        pos_class_rate = np.nan
        if task == 'PD':
            # Assuming binary classification where 1 is the default class
            # Calculate mean (proportion of 1s)
            pos_class_rate = np.mean(y)
        
        # Count numerical features
        n_num_features = 0
        N_path = task_dir / "N.npy"
        if N_path.exists():
            N = np.load(N_path, allow_pickle=False)
            if N.ndim == 2:
                n_num_features = N.shape[1]
            elif N.ndim == 1:
                n_num_features = 1
        
        # Count categorical features
        n_cat_features = 0
        C_path = task_dir / "C.npy"
        if C_path.exists():
            C = np.load(C_path, allow_pickle=False)
            if C.ndim == 2:
                n_cat_features = C.shape[1]
            elif C.ndim == 1:
                n_cat_features = 1
        
        # Total features BEFORE PCA
        n_features_raw = n_num_features + n_cat_features
        
        # Apply DataFeeder logic: if features > threshold, PCA reduces to target
        if n_features_raw > MAX_FEATURES_THRESHOLD:
            n_features_actual = PCA_TARGET_FEATURES
            pca_applied = True
        else:
            n_features_actual = n_features_raw
            pca_applied = False
        
        return {
            'dataset': dataset_name,
            'task': task,
            'n_rows': n_rows,
            'n_features': n_features_actual,  # Features AFTER potential PCA
            'n_features_raw': n_features_raw, # Features BEFORE PCA
            'n_num_features': n_num_features,
            'n_cat_features': n_cat_features,
            'pos_class_rate': pos_class_rate, # New characteristic: % of defaults
            'pca_applied': pca_applied,
            'dimensionality': n_rows * n_features_actual,
            'source': 'numpy_arrays'
        }
        
    except Exception as e:
        print(f"⚠️  Error loading numpy arrays for {dataset_name}: {e}")
        return None

# Collect info for all datasets
print("\n🔍 Processing datasets...")
found_count = 0
missing_count = 0

# --- Process PD ---
for dataset in all_datasets_pd:
    info = get_dataset_characteristics(dataset, 'PD')
    if info:
        dataset_info.append(info)
        found_count += 1
        pca_note = " [PCA→99]" if info['pca_applied'] else ""
        imbalance_str = f", {info['pos_class_rate']:.1%} defaults"
        print(f"  ✓ {dataset:30s} - {info['n_rows']:>8,} rows, {info['n_features']:>3} features ({info['n_num_features']} num + {info['n_cat_features']} cat){pca_note}{imbalance_str}")
    else:
        missing_count += 1
        print(f"  ✗ {dataset:30s} - NOT FOUND in processed directory")

# --- Process LGD ---
for dataset in all_datasets_lgd:
    info = get_dataset_characteristics(dataset, 'LGD')
    if info:
        dataset_info.append(info)
        found_count += 1
        pca_note = " [PCA→99]" if info['pca_applied'] else ""
        print(f"  ✓ {dataset:30s} - {info['n_rows']:>8,} rows, {info['n_features']:>3} features ({info['n_num_features']} num + {info['n_cat_features']} cat){pca_note}")
    else:
        missing_count += 1
        print(f"  ✗ {dataset:30s} - NOT FOUND in processed directory")

# Create DataFrame
if dataset_info:
    dataset_chars = pd.DataFrame(dataset_info)
    
    # Drop internal columns from display
    display_df = dataset_chars.drop(columns=['source', 'n_num_features', 'n_cat_features', 'n_features_raw', 'pca_applied'], errors='ignore')
    
    # Rename column for clarity
    display_df = display_df.rename(columns={'pos_class_rate': 'default_rate'})
    
    print(f"\n{'='*80}")
    print(f"✅ Successfully loaded {found_count} datasets")
    if missing_count > 0:
        print(f"⚠️  {missing_count} datasets not found (need preprocessing)")
    print(f"{'='*80}")
    
    # Show how many datasets had PCA applied
    n_pca = dataset_chars['pca_applied'].sum()
    if n_pca > 0:
        print(f"\n📊 {n_pca} datasets will use PCA (features > {MAX_FEATURES_THRESHOLD} → reduced to {PCA_TARGET_FEATURES})")
    
    print(f"\nDataset characteristics summary:")
    print(display_df[['n_rows', 'n_features', 'dimensionality']].describe())
    
    print(f"\nAll datasets (with default rates for PD):")
    # Format the default_rate column to percentage string for display
    display_df['default_rate'] = display_df['default_rate'].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else "-")
    print(display_df.to_string(index=False))
    
else:
    dataset_chars = pd.DataFrame()
    print(f"\n{'='*80}")
    print("⚠️  NO datasets found in processed directory!")
    print(f"{'='*80}")
    print("\n💡 Next steps:")
    print("   1. Make sure datasets have been preprocessed")
    print("   2. Check that processed files exist in:")
    print(f"      {PROCESSED_DATA_DIR}/{{task}}/{{dataset}}/")
    print("   3. Each dataset folder should contain: N.npy, C.npy, y.npy")

## C2. PD: Rank vs Dataset Characteristics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("\n" + "=" * 80)
print("  PD - RANK CORRELATION WITH DATASET CHARACTERISTICS")
print("=" * 80)

if not dataset_chars.empty and pivot_pd_hpo is not None:
    # Calculate ranks for each method across datasets
    ranks_df = pivot_pd_hpo.rank(axis=1, ascending=False, method='average')
    
    # Get PD datasets
    pd_chars = dataset_chars[dataset_chars['task'] == 'PD'].copy()
    
    # Calculate class imbalance (minority class proportion)
    if 'pos_class_rate' in pd_chars.columns:
        # Class imbalance = min(p, 1-p) where p is the positive class rate
        pd_chars['class_imbalance'] = pd_chars['pos_class_rate'].apply(
            lambda x: min(x, 1-x) if pd.notna(x) else np.nan
        )
    
    # Merge with dataset characteristics
    pd_chars = pd_chars.set_index('dataset')
    
    # Calculate correlations for each method
    correlations = []
    
    for method in ranks_df.columns:
        method_ranks = ranks_df[method]
        
        # Align datasets
        common_datasets = method_ranks.index.intersection(pd_chars.index)
        
        if len(common_datasets) > 3:  # Need at least 4 points for correlation
            aligned_ranks = method_ranks.loc[common_datasets]
            aligned_chars = pd_chars.loc[common_datasets]
            
            # 1. Size (n_rows)
            if aligned_chars['n_rows'].notna().sum() > 3:
                corr_size, p_size = stats.spearmanr(aligned_ranks, aligned_chars['n_rows'].fillna(aligned_chars['n_rows'].median()))
            else:
                corr_size, p_size = np.nan, np.nan
            
            # 2. Features (n_features)
            if aligned_chars['n_features'].notna().sum() > 3:
                corr_features, p_features = stats.spearmanr(aligned_ranks, aligned_chars['n_features'].fillna(aligned_chars['n_features'].median()))
            else:
                corr_features, p_features = np.nan, np.nan
            
            # 3. Dimensionality
            if aligned_chars['dimensionality'].notna().sum() > 3:
                corr_dim, p_dim = stats.spearmanr(aligned_ranks, aligned_chars['dimensionality'].fillna(aligned_chars['dimensionality'].median()))
            else:
                corr_dim, p_dim = np.nan, np.nan
                
            # 4. Class Imbalance (minority class proportion)
            if 'class_imbalance' in aligned_chars.columns and aligned_chars['class_imbalance'].notna().sum() > 3:
                corr_imbal, p_imbal = stats.spearmanr(aligned_ranks, aligned_chars['class_imbalance'].fillna(aligned_chars['class_imbalance'].median()))
            else:
                corr_imbal, p_imbal = np.nan, np.nan
            
            correlations.append({
                'method': method,
                'corr_size': corr_size, 'p_size': p_size,
                'corr_features': corr_features, 'p_features': p_features,
                'corr_dimensionality': corr_dim, 'p_dimensionality': p_dim,
                'corr_imbalance': corr_imbal, 'p_imbalance': p_imbal
            })
    
    corr_df = pd.DataFrame(correlations)
    
    if not corr_df.empty:
        print("\n📊 Spearman Correlations (Rank vs Dataset Characteristics)")
        print("   Positive correlation = higher rank (worse) as metric increases")
        print("   Negative correlation = lower rank (better) as metric increases\n")
        
        # Print table
        print(f"{'Method':<20} {'Size':>9} {'p-val':>7} {'Features':>9} {'p-val':>7} {'Dimension':>9} {'p-val':>7} {'Imbalance':>9} {'p-val':>7}")
        print("-" * 105)
        
        for _, row in corr_df.iterrows():
            sig_size = "*" if row['p_size'] < 0.05 else " "
            sig_feat = "*" if row['p_features'] < 0.05 else " "
            sig_dim = "*" if row['p_dimensionality'] < 0.05 else " "
            sig_imbal = "*" if row['p_imbalance'] < 0.05 else " "
            
            print(f"{row['method']:<20} "
                  f"{row['corr_size']:>8.3f}{sig_size} {row['p_size']:>7.3f} "
                  f"{row['corr_features']:>8.3f}{sig_feat} {row['p_features']:>7.3f} "
                  f"{row['corr_dimensionality']:>8.3f}{sig_dim} {row['p_dimensionality']:>7.3f} "
                  f"{row['corr_imbalance']:>8.3f}{sig_imbal} {row['p_imbalance']:>7.3f}")
        
        # =======================================================================
        # VISUALIZATION: 2x2 Grid of Bar Charts
        # =======================================================================
        fig, axes = plt.subplots(2, 2, figsize=(24, 14))
        
        # Helper function for plotting
        def plot_correlation_bar(ax, data, col_name, title):
            corr_sorted = data.sort_values(col_name, ascending=False)
            colors = ['red' if x > 0 else 'green' for x in corr_sorted[col_name]]
            ax.barh(range(len(corr_sorted)), corr_sorted[col_name], color=colors, alpha=0.7, edgecolor='black')
            ax.set_yticks(range(len(corr_sorted)))
            ax.set_yticklabels(corr_sorted['method'], fontsize=9)
            ax.set_xlabel('Spearman Correlation', fontweight='bold', fontsize=10)
            ax.set_title(title, fontweight='bold', fontsize=11, pad=10)
            ax.axvline(0, color='black', linewidth=1.5)
            ax.grid(axis='x', alpha=0.3)
            
            # Add value labels
            for i, (idx, row) in enumerate(corr_sorted.iterrows()):
                val = row[col_name]
                if pd.notna(val):
                    ax.text(val + 0.02 if val > 0 else val - 0.02, i, f'{val:.2f}', 
                           va='center', ha='left' if val > 0 else 'right', fontsize=8)

        # Plot 1: Size (Top Left)
        plot_correlation_bar(axes[0, 0], corr_df, 'corr_size', 
                             'Rank vs Dataset Size\n(red=worse on large, green=better)')
        
        # Plot 2: Features (Top Middle)
        plot_correlation_bar(axes[0, 1], corr_df, 'corr_features', 
                             'Rank vs Feature Count\n(red=worse on high-dim, green=better)')
        
        # Plot 3: Dimensionality (Top Right)
        plot_correlation_bar(axes[1, 0], corr_df, 'corr_dimensionality', 
                             'Rank vs Dimensionality\n(red=worse on complex, green=better)')
                             
        # Plot 4: Class Imbalance (Bottom Left)
        plot_correlation_bar(axes[1, 1], corr_df, 'corr_imbalance', 
                             'Rank vs Class Imbalance\n(red=worse when imbalanced, green=better)')

        
    else:
        print("⚠️  Not enough data for correlation analysis")
else:
    print("⚠️  Dataset characteristics or HPO results not available")

## C3. PD: Rank vs Dataset Characteristics per method

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats

# ==============================================================================
# CONFIGURATION
# ==============================================================================
METHODS_TO_ANALYZE = ['tabpfn', 'tabpfn_v2', 'catboost', 'tabicl']

print("\n" + "="*80)
print("  PD - RANK SCATTER PLOTS (Dataset-Level Ranks)")
print("="*80)

if 'pd_agg' not in locals() or 'dataset_chars' not in locals():
    print("⚠️  Error: 'pd_agg' or 'dataset_chars' not found. Please run the previous data loading cells.")
else:
    # ==============================================================================
    # USE THE EXACT SAME RANKING METHODOLOGY AS THE HEATMAP
    # ==============================================================================
    
    # 1. Filter for HPO mode only (same as heatmap)
    pd_agg_hpo = pd_agg[pd_agg['hpo_mode'] == 'HPO'].copy()
    
    # 2. Identify the metric column
    metric_col = 'AUC_mean'  # Aggregated mean AUC per dataset
    
    if metric_col not in pd_agg_hpo.columns:
        print(f"⚠️  Error: '{metric_col}' not found in pd_agg")
        print(f"   Available columns: {list(pd_agg_hpo.columns)}")
    else:
        print(f"✓ Using aggregated data with metric: '{metric_col}'")
        
        # 3. Create pivot table (same as heatmap)
        pivot = pd_agg_hpo.pivot(index='dataset', columns='method', values=metric_col)
        
        print(f"✓ Pivot table shape: {pivot.shape}")
        print(f"  - Datasets: {len(pivot)}")
        print(f"  - Methods: {len(pivot.columns)}")
        
        # 4. Calculate ranks (EXACT SAME as heatmap: rank across methods for each dataset)
        ranks_df = pivot.rank(axis=1, ascending=False, method='average')
        
        # 5. Convert ranks to long format for plotting
        ranks_long = ranks_df.reset_index().melt(
            id_vars='dataset', 
            var_name='method', 
            value_name='rank'
        )
        
        print(f"✓ Ranks calculated for {len(ranks_long)} dataset-method combinations")
        
        # 6. Prepare dataset characteristics
        pd_chars = dataset_chars[dataset_chars['task'] == 'PD'].copy()
        
        # Calculate class imbalance (minority class proportion)
        if 'pos_class_rate' in pd_chars.columns:
            pd_chars['class_imbalance'] = pd_chars['pos_class_rate'].apply(
                lambda x: min(x, 1-x) if pd.notna(x) else np.nan
            )
        
        # 7. Merge ranks with dataset characteristics
        plot_data = pd.merge(ranks_long, pd_chars, on='dataset', how='inner')
        
        print(f"✓ Merged data: {len(plot_data)} dataset-method combinations")
        print(f"  - Datasets with characteristics: {plot_data['dataset'].nunique()}")
        
        # Filter for the specific methods requested
        valid_methods = [m for m in METHODS_TO_ANALYZE if m in plot_data['method'].unique()]
        print(f"✓ Methods to plot: {valid_methods}\n")
        
        # Print diagnostic info for each method
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method]
            if not method_data.empty:
                print(f"  {method}:")
                print(f"    - Datasets: {len(method_data)}")
                print(f"    - Rank range: {method_data['rank'].min():.2f} - {method_data['rank'].max():.2f}")
                print(f"    - Rank 1 count: {(method_data['rank'] == 1.0).sum()} datasets")
                print(f"    - Average rank: {method_data['rank'].mean():.2f}")
        
        # ==============================================================================
        # GENERATE PLOTS
        # ==============================================================================
        sns.set_style("whitegrid")
        
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method].copy()
            if method_data.empty: 
                print(f"⚠️  No data for method: {method}")
                continue
            
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle(f"Method: {method} - Dataset-Level Rank (Lower is Better)", 
                        fontsize=18, fontweight='bold', y=0.98)
            
            # Helper for clean plotting
            def plot_clean(ax, x_col, label, log_scale=False):
                if x_col not in method_data.columns: 
                    ax.text(0.5, 0.5, f'No {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Remove NaN values for this specific plot
                plot_subset = method_data[[x_col, 'rank', 'dataset']].dropna()
                
                if len(plot_subset) == 0:
                    ax.text(0.5, 0.5, f'No valid {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Scatter: Blue dots
                scatter = ax.scatter(
                    plot_subset[x_col], plot_subset['rank'],
                    color='tab:blue', s=80, alpha=0.7, edgecolor='darkblue', linewidth=1.5
                )
                
                # Trendline
                try:
                    sns.regplot(
                        data=plot_subset, x=x_col, y='rank',
                        ax=ax, scatter=False, color='red',
                        line_kws={'linestyle': '--', 'linewidth': 2, 'alpha': 0.7}
                    )
                except Exception as e:
                    print(f"  ⚠️  Could not fit trendline for {label}: {e}")
                
                # Formatting
                ax.set_xlabel(label, fontsize=12, fontweight='bold')
                ax.set_ylabel('Rank (Lower = Better)', fontsize=12, fontweight='bold')
                
                # Y-axis: Start from 0.5, go up to max rank + 0.5
                y_max = plot_data['rank'].max() + 0.5
                ax.set_ylim(0.5, y_max)
                
                # Log scale for x if requested
                if log_scale:
                    ax.set_xscale('log')
                
                # Grid
                ax.grid(True, linestyle=':', alpha=0.4, linewidth=0.8)

            # Plot 1: Class Imbalance (Top Left)
            if 'class_imbalance' in method_data.columns:
                plot_clean(axes[0,0], 'class_imbalance', 'Class Imbalance (Minority Proportion)', log_scale=False)
                axes[0,0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
                axes[0,0].set_title("Rank vs Class Imbalance", fontsize=13, pad=10, fontweight='bold')
            else:
                axes[0,0].text(0.5, 0.5, 'No class imbalance data', ha='center', va='center', 
                              transform=axes[0,0].transAxes, fontsize=12)

            # Plot 2: Rows (Top Right) - Log scale
            plot_clean(axes[0,1], 'n_rows', 'Number of Rows (Log Scale)', log_scale=True)
            axes[0,1].set_title("Rank vs Dataset Size", fontsize=13, pad=10, fontweight='bold')

            # Plot 3: Features (Bottom Left) - Log scale
            plot_clean(axes[1,0], 'n_features', 'Number of Features (Log Scale)', log_scale=True)
            axes[1,0].set_title("Rank vs Feature Count", fontsize=13, pad=10, fontweight='bold')

            # Plot 4: Dimensionality (Bottom Right) - Log scale
            plot_clean(axes[1,1], 'dimensionality', 'Dimensionality (Log Scale)', log_scale=True)
            axes[1,1].set_title("Rank vs Dimensionality (Rows × Features)", fontsize=13, pad=10, fontweight='bold')

            plt.tight_layout(rect=[0, 0, 1, 0.96])
            
            filename = FIGURES_DIR / f"pd_rank_scatter_{method}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"\n✅ Saved: {filename.name}")
            plt.show()
            
        print("\n" + "="*80)
        print("✅ All scatter plots generated successfully!")
        print("  These plots now use EXACTLY the same ranking as the heatmap:")
        print("  - Ranks based on mean AUC per dataset")
        print("  - Each dot = one dataset")
        print("  - Rank 1 dots should match heatmap exactly")
        print("="*80)

## C4. LGD: Rank vs Dataset Characteristics

In [ ]:
print("\n" + "=" * 80)
print("  LGD - RANK CORRELATION WITH DATASET CHARACTERISTICS")
print("=" * 80)

if not dataset_chars.empty and pivot_lgd_hpo is not None:
    # Calculate ranks for each method across datasets
    ranks_df = pivot_lgd_hpo.rank(axis=1, ascending=False, method='average')
    
    # Get LGD datasets
    lgd_chars = dataset_chars[dataset_chars['task'] == 'LGD'].copy()
    
    # Merge with dataset characteristics
    lgd_chars = lgd_chars.set_index('dataset')
    
    # Calculate correlations for each method
    correlations = []
    
    for method in ranks_df.columns:
        method_ranks = ranks_df[method]
        
        # Align datasets
        common_datasets = method_ranks.index.intersection(lgd_chars.index)
        
        if len(common_datasets) > 3:  # Need at least 4 points for correlation
            aligned_ranks = method_ranks.loc[common_datasets]
            aligned_chars = lgd_chars.loc[common_datasets]
            
            # Calculate Spearman correlations
            if aligned_chars['n_rows'].notna().sum() > 3:
                corr_size, p_size = stats.spearmanr(aligned_ranks, aligned_chars['n_rows'].fillna(aligned_chars['n_rows'].median()))
            else:
                corr_size, p_size = np.nan, np.nan
            
            if aligned_chars['n_features'].notna().sum() > 3:
                corr_features, p_features = stats.spearmanr(aligned_ranks, aligned_chars['n_features'].fillna(aligned_chars['n_features'].median()))
            else:
                corr_features, p_features = np.nan, np.nan
            
            if aligned_chars['dimensionality'].notna().sum() > 3:
                corr_dim, p_dim = stats.spearmanr(aligned_ranks, aligned_chars['dimensionality'].fillna(aligned_chars['dimensionality'].median()))
            else:
                corr_dim, p_dim = np.nan, np.nan
            
            correlations.append({
                'method': method,
                'corr_size': corr_size,
                'p_size': p_size,
                'corr_features': corr_features,
                'p_features': p_features,
                'corr_dimensionality': corr_dim,
                'p_dimensionality': p_dim
            })
    
    corr_df = pd.DataFrame(correlations)
    
    if not corr_df.empty:
        print("\n📊 Spearman Correlations (Rank vs Dataset Characteristics)")
        print("   Positive correlation = higher rank (worse) on larger/more complex datasets")
        print("   Negative correlation = lower rank (better) on larger/more complex datasets\n")
        
        print(f"{'Method':<20} {'Size':>10} {'p-val':>8} {'Features':>10} {'p-val':>8} {'Dimension':>10} {'p-val':>8}")
        print("-" * 88)
        
        for _, row in corr_df.iterrows():
            sig_size = "*" if row['p_size'] < 0.05 else " "
            sig_feat = "*" if row['p_features'] < 0.05 else " "
            sig_dim = "*" if row['p_dimensionality'] < 0.05 else " "
            
            print(f"{row['method']:<20} {row['corr_size']:>9.3f}{sig_size} {row['p_size']:>8.3f} "
                  f"{row['corr_features']:>9.3f}{sig_feat} {row['p_features']:>8.3f} "
                  f"{row['corr_dimensionality']:>9.3f}{sig_dim} {row['p_dimensionality']:>8.3f}")
        
        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(24, 6))
        
        # Plot 1: Size
        corr_sorted = corr_df.sort_values('corr_size', ascending=False)
        colors = ['red' if x > 0 else 'green' for x in corr_sorted['corr_size']]
        axes[0].barh(range(len(corr_sorted)), corr_sorted['corr_size'], color=colors, alpha=0.7, edgecolor='black')
        axes[0].set_yticks(range(len(corr_sorted)))
        axes[0].set_yticklabels(corr_sorted['method'])
        axes[0].set_xlabel('Spearman Correlation', fontweight='bold')
        axes[0].set_title('LGD: Rank vs Dataset Size\n(red=worse on large, green=better on large)', fontweight='bold')
        axes[0].axvline(0, color='black', linewidth=1)
        axes[0].grid(axis='x', alpha=0.3)
        
        # Plot 2: Features
        corr_sorted = corr_df.sort_values('corr_features', ascending=False)
        colors = ['red' if x > 0 else 'green' for x in corr_sorted['corr_features']]
        axes[1].barh(range(len(corr_sorted)), corr_sorted['corr_features'], color=colors, alpha=0.7, edgecolor='black')
        axes[1].set_yticks(range(len(corr_sorted)))
        axes[1].set_yticklabels(corr_sorted['method'])
        axes[1].set_xlabel('Spearman Correlation', fontweight='bold')
        axes[1].set_title('LGD: Rank vs Number of Features\n(red=worse on high-dim, green=better on high-dim)', fontweight='bold')
        axes[1].axvline(0, color='black', linewidth=1)
        axes[1].grid(axis='x', alpha=0.3)
        
        # Plot 3: Dimensionality
        corr_sorted = corr_df.sort_values('corr_dimensionality', ascending=False)
        colors = ['red' if x > 0 else 'green' for x in corr_sorted['corr_dimensionality']]
        axes[2].barh(range(len(corr_sorted)), corr_sorted['corr_dimensionality'], color=colors, alpha=0.7, edgecolor='black')
        axes[2].set_yticks(range(len(corr_sorted)))
        axes[2].set_yticklabels(corr_sorted['method'])
        axes[2].set_xlabel('Spearman Correlation', fontweight='bold')
        axes[2].set_title('LGD: Rank vs Dimensionality (rows × cols)\n(red=worse on complex, green=better on complex)', fontweight='bold')
        axes[2].axvline(0, color='black', linewidth=1)
        axes[2].grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_rank_correlation_dataset_chars.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"\n✅ Saved: {filename.name}")
        plt.show()
    else:
        print("⚠️  Not enough data for correlation analysis")
else:
    print("⚠️  Dataset characteristics or HPO results not available")

## C5. LGD: Rank vs Dataset Characteristics per method

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats

# ==============================================================================
# CONFIGURATION
# ==============================================================================
METHODS_TO_ANALYZE = ['tabpfn', 'tabpfn_v2', 'catboost', 'tabicl']

print("\n" + "="*80)
print("  LGD - RANK SCATTER PLOTS (Dataset-Level Ranks)")
print("="*80)

if 'lgd_agg' not in locals() or 'dataset_chars' not in locals():
    print("⚠️  Error: 'lgd_agg' or 'dataset_chars' not found. Please run the previous data loading cells.")
else:
    # ==============================================================================
    # USE THE EXACT SAME RANKING METHODOLOGY AS THE HEATMAP
    # ==============================================================================
    
    # 1. Filter for HPO mode only (same as heatmap)
    lgd_agg_hpo = lgd_agg[lgd_agg['hpo_mode'] == 'HPO'].copy()
    
    # 2. Identify the metric column
    metric_col = 'R2_mean'  # Aggregated mean R² per dataset
    
    if metric_col not in lgd_agg_hpo.columns:
        print(f"⚠️  Error: '{metric_col}' not found in lgd_agg")
        print(f"   Available columns: {list(lgd_agg_hpo.columns)}")
    else:
        print(f"✓ Using aggregated data with metric: '{metric_col}'")
        
        # 3. Create pivot table (same as heatmap)
        pivot = lgd_agg_hpo.pivot(index='dataset', columns='method', values=metric_col)
        
        print(f"✓ Pivot table shape: {pivot.shape}")
        print(f"  - Datasets: {len(pivot)}")
        print(f"  - Methods: {len(pivot.columns)}")
        
        # 4. Calculate ranks (EXACT SAME as heatmap: rank across methods for each dataset)
        ranks_df = pivot.rank(axis=1, ascending=False, method='average')
        
        # 5. Convert ranks to long format for plotting
        ranks_long = ranks_df.reset_index().melt(
            id_vars='dataset', 
            var_name='method', 
            value_name='rank'
        )
        
        print(f"✓ Ranks calculated for {len(ranks_long)} dataset-method combinations")
        
        # 6. Prepare dataset characteristics (LGD only)
        lgd_chars = dataset_chars[dataset_chars['task'] == 'LGD'].copy()
        
        # 7. Merge ranks with dataset characteristics
        plot_data = pd.merge(ranks_long, lgd_chars, on='dataset', how='inner')
        
        print(f"✓ Merged data: {len(plot_data)} dataset-method combinations")
        print(f"  - Datasets with characteristics: {plot_data['dataset'].nunique()}")
        
        # Filter for the specific methods requested
        valid_methods = [m for m in METHODS_TO_ANALYZE if m in plot_data['method'].unique()]
        print(f"✓ Methods to plot: {valid_methods}\n")
        
        # Print diagnostic info for each method
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method]
            if not method_data.empty:
                print(f"  {method}:")
                print(f"    - Datasets: {len(method_data)}")
                print(f"    - Rank range: {method_data['rank'].min():.2f} - {method_data['rank'].max():.2f}")
                print(f"    - Rank 1 count: {(method_data['rank'] == 1.0).sum()} datasets")
                print(f"    - Average rank: {method_data['rank'].mean():.2f}")
        
        # ==============================================================================
        # GENERATE PLOTS (3 plots: rows, features, dimensionality - NO class imbalance)
        # ==============================================================================
        sns.set_style("whitegrid")
        
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method].copy()
            if method_data.empty: 
                print(f"⚠️  No data for method: {method}")
                continue
            
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(f"Method: {method} - Dataset-Level Rank (Lower is Better)", 
                        fontsize=18, fontweight='bold', y=1.02)
            
            # Helper for clean plotting
            def plot_clean(ax, x_col, label, log_scale=False):
                if x_col not in method_data.columns: 
                    ax.text(0.5, 0.5, f'No {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Remove NaN values for this specific plot
                plot_subset = method_data[[x_col, 'rank', 'dataset']].dropna()
                
                if len(plot_subset) == 0:
                    ax.text(0.5, 0.5, f'No valid {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Scatter: Blue dots
                scatter = ax.scatter(
                    plot_subset[x_col], plot_subset['rank'],
                    color='tab:blue', s=80, alpha=0.7, edgecolor='darkblue', linewidth=1.5
                )
                
                # Trendline
                try:
                    sns.regplot(
                        data=plot_subset, x=x_col, y='rank',
                        ax=ax, scatter=False, color='red',
                        line_kws={'linestyle': '--', 'linewidth': 2, 'alpha': 0.7}
                    )
                except Exception as e:
                    print(f"  ⚠️  Could not fit trendline for {label}: {e}")
                
                # Formatting
                ax.set_xlabel(label, fontsize=12, fontweight='bold')
                ax.set_ylabel('Rank (Lower = Better)', fontsize=12, fontweight='bold')
                
                # Y-axis: Start from 0.5, go up to max rank + 0.5
                y_max = plot_data['rank'].max() + 0.5
                ax.set_ylim(0.5, y_max)
                
                # Log scale for x if requested
                if log_scale:
                    ax.set_xscale('log')
                
                # Grid
                ax.grid(True, linestyle=':', alpha=0.4, linewidth=0.8)

            # Plot 1: Rows (Left) - Log scale
            plot_clean(axes[0], 'n_rows', 'Number of Rows (Log Scale)', log_scale=True)
            axes[0].set_title("Rank vs Dataset Size", fontsize=13, pad=10, fontweight='bold')

            # Plot 2: Features (Middle) - Log scale
            plot_clean(axes[1], 'n_features', 'Number of Features (Log Scale)', log_scale=True)
            axes[1].set_title("Rank vs Feature Count", fontsize=13, pad=10, fontweight='bold')

            # Plot 3: Dimensionality (Right) - Log scale
            plot_clean(axes[2], 'dimensionality', 'Dimensionality (Log Scale)', log_scale=True)
            axes[2].set_title("Rank vs Dimensionality (Rows × Features)", fontsize=13, pad=10, fontweight='bold')

            plt.tight_layout(rect=[0, 0, 1, 0.98])
            
            filename = FIGURES_DIR / f"lgd_rank_scatter_{method}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"\n✅ Saved: {filename.name}")
            plt.show()
            
        print("\n" + "="*80)
        print("✅ All scatter plots generated successfully!")
        print("  These plots now use EXACTLY the same ranking as the heatmap:")
        print("  - Ranks based on mean R² per dataset")
        print("  - Each dot = one dataset")
        print("  - Rank 1 dots should match heatmap exactly")
        print("="*80)

---
# ⏱️ Part D: Training Time Analysis

Analyze computational efficiency of methods:
- **Training time matrices**: Dataset × Method training times
- **Average times**: Method efficiency comparison
- **HPO overhead**: Additional cost of hyperparameter tuning

## D1. PD Training Time Analysis

In [ ]:
print("\n" + "=" * 80)
print("  PD - TRAINING TIME ANALYSIS (NO_HPO)")
print("=" * 80)

# Helper function to format time
def format_time(seconds):
    """Format time as minutes or seconds with appropriate suffix."""
    if seconds >= 60:
        minutes = seconds / 60
        return f"{minutes:.2f}m"
    else:
        return f"{seconds:.2f}s"

# Check if training time data is available in raw results
if 'train_time' in pd_raw.columns:
    # Calculate TOTAL training time per method-dataset (sum across all folds)
    time_summary = pd_raw.groupby(['dataset', 'method', 'hpo_mode'])['train_time'].sum().reset_index()
    time_summary.rename(columns={'train_time': 'total_train_time'}, inplace=True)
    
    # Filter for NO_HPO only
    df_no_hpo = time_summary[time_summary['hpo_mode'] == 'NO_HPO'].copy()
    
    if not df_no_hpo.empty:
        # =======================================================================
        # TRAINING TIME HEATMAP
        # =======================================================================
        print("\nCreating PD training time heatmap (NO_HPO)...")
        
        pivot_time = df_no_hpo.pivot(index='dataset', columns='method', values='total_train_time')
        pivot_time = pivot_time.sort_index()
        
        # Sort methods by average training time (fastest to slowest)
        method_means = pivot_time.mean(axis=0).sort_values()
        pivot_time = pivot_time[method_means.index]
        
        # Create formatted version for display
        pivot_time_formatted = pivot_time.applymap(format_time)
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            pivot_time, annot=pivot_time_formatted, fmt='', cmap='YlOrRd',
            cbar_kws={'label': 'Total Training Time (seconds)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('PD: Total Training Time per Dataset-Method (NO_HPO)\nDatasets × Methods (sum across all folds)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "pd_training_time_total_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # TRAINING TIME RANK MATRIX
        # =======================================================================
        print("\nCreating PD training time rank matrix (NO_HPO)...")
        
        # Calculate ranks (1 = fastest)
        rank_time = pivot_time.rank(axis=1, method='average')
        
        # Sort by average rank
        method_avg_ranks = rank_time.mean(axis=0).sort_values()
        rank_time = rank_time[method_avg_ranks.index]
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            rank_time, annot=True, fmt='.1f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Rank (1=fastest)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('PD: Training Time Ranks per Dataset-Method (NO_HPO)\nDatasets × Methods (1=fastest)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "pd_training_time_ranks_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # SUMMARY STATISTICS
        # =======================================================================
        print("\n" + "=" * 80)
        print("  TRAINING TIME SUMMARY (NO_HPO)")
        print("=" * 80)
        
        print("\nAverage Total Training Time per Method:")
        method_totals = df_no_hpo.groupby('method')['total_train_time'].mean().sort_values()
        for method, time_val in method_totals.items():
            print(f"  {method:20s}: {format_time(time_val):>10s}")
        
        print("\n" + "=" * 80)
        
    else:
        print("⚠️  No NO_HPO data available")
        
else:
    print("⚠️  Training time data not available in raw results")
    print("     Expected column: 'train_time'")
    print(f"     Available columns: {list(pd_raw.columns)}")

## D2. LGD Training Time Analysis

In [ ]:
print("\n" + "=" * 80)
print("  LGD - TRAINING TIME ANALYSIS (NO_HPO)")
print("=" * 80)

# Helper function to format time
def format_time(seconds):
    """Format time as minutes or seconds with appropriate suffix."""
    if seconds >= 60:
        minutes = seconds / 60
        return f"{minutes:.2f}m"
    else:
        return f"{seconds:.2f}s"

# Check if training time data is available in raw results
if 'train_time' in lgd_raw.columns:
    # Calculate TOTAL training time per method-dataset (sum across all folds)
    time_summary = lgd_raw.groupby(['dataset', 'method', 'hpo_mode'])['train_time'].sum().reset_index()
    time_summary.rename(columns={'train_time': 'total_train_time'}, inplace=True)
    
    # Filter for NO_HPO only
    df_no_hpo = time_summary[time_summary['hpo_mode'] == 'NO_HPO'].copy()
    
    if not df_no_hpo.empty:
        # =======================================================================
        # TRAINING TIME HEATMAP
        # =======================================================================
        print("\nCreating LGD training time heatmap (NO_HPO)...")
        
        pivot_time = df_no_hpo.pivot(index='dataset', columns='method', values='total_train_time')
        pivot_time = pivot_time.sort_index()
        
        # Sort methods by average training time (fastest to slowest)
        method_means = pivot_time.mean(axis=0).sort_values()
        pivot_time = pivot_time[method_means.index]
        
        # Create formatted version for display
        pivot_time_formatted = pivot_time.applymap(format_time)
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            pivot_time, annot=pivot_time_formatted, fmt='', cmap='YlOrRd',
            cbar_kws={'label': 'Total Training Time (seconds)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('LGD: Total Training Time per Dataset-Method (NO_HPO)\nDatasets × Methods (sum across all folds)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_training_time_total_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # TRAINING TIME RANK MATRIX
        # =======================================================================
        print("\nCreating LGD training time rank matrix (NO_HPO)...")
        
        # Calculate ranks (1 = fastest)
        rank_time = pivot_time.rank(axis=1, method='average')
        
        # Sort by average rank
        method_avg_ranks = rank_time.mean(axis=0).sort_values()
        rank_time = rank_time[method_avg_ranks.index]
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            rank_time, annot=True, fmt='.1f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Rank (1=fastest)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('LGD: Training Time Ranks per Dataset-Method (NO_HPO)\nDatasets × Methods (1=fastest)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_training_time_ranks_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # SUMMARY STATISTICS
        # =======================================================================
        print("\n" + "=" * 80)
        print("  TRAINING TIME SUMMARY (NO_HPO)")
        print("=" * 80)
        
        print("\nAverage Total Training Time per Method:")
        method_totals = df_no_hpo.groupby('method')['total_train_time'].mean().sort_values()
        for method, time_val in method_totals.items():
            print(f"  {method:20s}: {format_time(time_val):>10s}")
        
        print("\n" + "=" * 80)
        
    else:
        print("⚠️  No NO_HPO data available")
        
else:
    print("⚠️  Training time data not available in raw results")
    print("     Expected column: 'train_time'")
    print(f"     Available columns: {list(lgd_raw.columns)}")

---
## Summary

This notebook analyzed Experiment1 results with comprehensive visualizations:

**For both PD and LGD tasks:**
- Performance heatmaps (NO_HPO, HPO, improvement)
- Average performance bar charts with error bars
- Performance distribution boxplots across folds
- Rank heatmaps and average rank bar charts
- PAMA analysis at fold level (not dataset level)
- Rank correlation with dataset characteristics (size, features, dimensionality)
- **Training time analysis** (heatmaps, averages)

All visualizations saved to `figures/` directory at 300 DPI.